In [1]:
import pandas as pd, numpy as np, time, warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE            # если классы несбалансированы
warnings.filterwarnings('ignore')

2. Загрузка датасета Kaggle ((https://www.kaggle.com/datasets/eswarchandt/phishing-website-detector?resource=download))


In [2]:
DATA_PATH = "phishing.csv"    
df = pd.read_csv(DATA_PATH)

print(df.columns.tolist())


['Index', 'UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon', 'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage', 'StatsReport', 'class']


3. Подготовка

In [3]:
target_col = 'class'
y_raw = df[target_col]

# переводим метку в 0/1: 1 = phishing, 0 = нормально
y = y_raw.replace({-1: 1,   # phishing
                   0 : 1,   # suspicious -> тоже считаем фишингом (можно иначе)
                   1 : 0})  # legitimate

X = df.drop(columns=[target_col, 'Index'])    # убираем Index


X_bal, y_bal = SMOTE(random_state=42).fit_resample(X, y)


X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=42
)

4. Определяем модели

In [6]:
models = {
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective='binary:logistic', eval_metric='logloss',
        n_jobs=-1, random_state=42
    )
}

summary = []

5. Обучение, проверка, кросс-валидация

In [7]:
for name, mdl in models.items():
    t0 = time.time();           mdl.fit(X_tr, y_tr)
    fit_time = round(time.time() - t0, 2)

    y_pred  = mdl.predict(X_te)
    y_proba = mdl.predict_proba(X_te)[:, 1]

    summary.append({
        "Model"     : name,
        "Accuracy"  : accuracy_score(y_te, y_pred),
        "Precision" : precision_score(y_te, y_pred),
        "Recall"    : recall_score(y_te, y_pred),
        "F1"        : f1_score(y_te, y_pred),
        "ROC-AUC"   : roc_auc_score(y_te, y_proba),
        "Fit s"     : fit_time,
        "CV AUC (5)": cross_val_score(
            mdl, X_bal, y_bal, cv=5, scoring='roc_auc').mean()
    })

    print(f"\n{name} — отчёт hold-out:\n",
          classification_report(y_te, y_pred, digits=3))

print("\n=== Сводка ===")
print(pd.DataFrame(summary).set_index("Model").round(3))


GradientBoosting — отчёт hold-out:
               precision    recall  f1-score   support

           0      0.950     0.957     0.953      1232
           1      0.957     0.950     0.953      1231

    accuracy                          0.953      2463
   macro avg      0.953     0.953     0.953      2463
weighted avg      0.953     0.953     0.953      2463


XGBoost — отчёт hold-out:
               precision    recall  f1-score   support

           0      0.966     0.978     0.972      1232
           1      0.978     0.966     0.972      1231

    accuracy                          0.972      2463
   macro avg      0.972     0.972     0.972      2463
weighted avg      0.972     0.972     0.972      2463


=== Сводка ===
                  Accuracy  Precision  Recall     F1  ROC-AUC  Fit s  \
Model                                                                  
GradientBoosting     0.953      0.957   0.950  0.953    0.994   5.75   
XGBoost              0.972      0.978   0.966  0.